# Recommendation Baselines with Cold-Start Evaluation

This notebook evaluates two baseline recommenders on **MovieLens** and **Last.fm**:

- **Popularity**
- **Matrix Factorization**

Cold-start follows the assignment requirement:

1. First split the **full dataset** into train/test.
2. Then count each user's interactions **inside the training set**.
3. Users with **fewer than 5 training interactions** are treated as **cold-start users**.
4. These cold-start users are **removed from model training**.
5. Evaluation is reported **separately** for:
   - **Regular users**
   - **Cold-start users**

For MF, regular users are scored with the trained user embedding.  
For cold-start users, a temporary user embedding is inferred from their limited history while keeping the trained item factors fixed.


In [2]:
import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

SEED = 42
K = 10
NUM_NEGATIVES = 99
COLD_USER_THRESHOLD = 5   # cold-start users: training interactions < 5
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BASE_DIR = Path.cwd()

print('DEVICE =', DEVICE)

DEVICE = cuda


In [3]:

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)


def build_id_mappings(values):
    uniq = pd.Index(sorted(pd.unique(values)))
    val2idx = {v: i for i, v in enumerate(uniq)}
    idx2val = {i: v for i, v in enumerate(uniq)}
    return uniq, val2idx, idx2val


def encode_series(series, mapping):
    return series.map(mapping).astype(np.int64)


def make_random_timestamps_for_lastfm(df, seed=42):
    rng = np.random.default_rng(seed)
    start_ts = 1262304000   # 2010-01-01
    end_ts   = 1609372800   # 2020-12-31
    df = df.copy()
    df['timestamp'] = rng.integers(start_ts, end_ts, size=len(df), endpoint=False)
    return df


def sample_negatives(all_items, seen_items, positive_item, rng, num_negatives=99):
    forbidden = set(seen_items) | {positive_item}
    candidates = np.array(sorted(set(all_items) - forbidden), dtype=np.int64)
    if len(candidates) < num_negatives:
        return None
    negatives = rng.choice(candidates, size=num_negatives, replace=False)
    return np.concatenate([[positive_item], negatives])


def evaluate_sampled_ranking(score_fn, users, positive_items, candidate_items, k=10, batch_size=256):
    users = [u for u in users if u in positive_items and u in candidate_items]
    if len(users) == 0:
        return {f'HR@{k}': np.nan, f'NDCG@{k}': np.nan, 'MAP': np.nan, 'num_eval_users': 0}

    hr_total = 0.0
    ndcg_total = 0.0
    map_total = 0.0

    for start in range(0, len(users), batch_size):
        batch_users = users[start:start + batch_size]
        score_matrix = score_fn(batch_users)

        for row, u in enumerate(batch_users):
            pos_item = int(positive_items[u])
            cands = candidate_items[u]
            cand_scores = score_matrix[row, cands]
            order = np.argsort(-cand_scores, kind='stable')
            ranked_items = cands[order]
            rank = int(np.flatnonzero(ranked_items == pos_item)[0]) + 1

            hr_total += float(rank <= k)
            ndcg_total += float(1.0 / np.log2(rank + 1)) if rank <= k else 0.0
            map_total += float(1.0 / rank)

    n = len(users)
    return {
        f'HR@{k}': hr_total / n,
        f'NDCG@{k}': ndcg_total / n,
        'MAP': map_total / n,
        'num_eval_users': n,
    }


def split_movielens_leave_one_positive(df, rating_threshold=4.0):
    train_parts = []
    test_positive_raw = {}
    user_full_seen_raw = {}

    dropped_no_positive = 0
    dropped_no_history_before_test = 0

    for user, g in df.groupby('user', sort=False):
        g = g.sort_values(['timestamp', 'item']).reset_index(drop=True)
        user_full_seen_raw[int(user)] = set(g['item'].astype(int).tolist())

        pos_positions = np.flatnonzero(g['value'].to_numpy(dtype=np.float32) >= rating_threshold)
        if len(pos_positions) == 0:
            dropped_no_positive += 1
            continue

        test_pos_loc = int(pos_positions[-1])
        if test_pos_loc == 0:
            dropped_no_history_before_test += 1
            continue

        train_parts.append(g.iloc[:test_pos_loc].copy())
        test_positive_raw[int(user)] = int(g.iloc[test_pos_loc]['item'])

    train_all = pd.concat(train_parts, ignore_index=True) if train_parts else pd.DataFrame(columns=df.columns)

    meta = {
        'dropped_no_positive': dropped_no_positive,
        'dropped_no_history_before_test': dropped_no_history_before_test,
    }
    return train_all, test_positive_raw, user_full_seen_raw, meta


def split_lastfm_leave_one_out(df):
    train_parts = []
    test_positive_raw = {}
    user_full_seen_raw = {}
    dropped_short_history = 0

    for user, g in df.groupby('user', sort=False):
        g = g.sort_values(['timestamp', 'item']).reset_index(drop=True)
        user_full_seen_raw[int(user)] = set(g['item'].astype(int).tolist())

        if len(g) < 2:
            dropped_short_history += 1
            continue

        train_parts.append(g.iloc[:-1].copy())
        test_positive_raw[int(user)] = int(g.iloc[-1]['item'])

    train_all = pd.concat(train_parts, ignore_index=True) if train_parts else pd.DataFrame(columns=df.columns)

    meta = {
        'dropped_short_history': dropped_short_history,
    }
    return train_all, test_positive_raw, user_full_seen_raw, meta


def prepare_cold_start_protocol(train_all, test_positive_raw, user_full_seen_raw, num_negatives=99, seed=42):
    # 1) identify cold users from TRAINING SET ONLY
    train_user_counts = train_all.groupby('user').size()
    cold_users_raw = set(train_user_counts[train_user_counts < COLD_USER_THRESHOLD].index.astype(int))
    regular_users_raw = set(train_user_counts[train_user_counts >= COLD_USER_THRESHOLD].index.astype(int))

    # 2) remove cold users from training
    train_regular = train_all[~train_all['user'].isin(cold_users_raw)].copy()

    # 3) build mappings only from regular-user training data
    user_ids, user2idx, idx2user = build_id_mappings(train_regular['user'])
    item_ids, item2idx, idx2item = build_id_mappings(train_regular['item'])

    train_regular['user_idx'] = encode_series(train_regular['user'], user2idx)
    train_regular['item_idx'] = encode_series(train_regular['item'], item2idx)

    all_items = np.arange(len(item_ids), dtype=np.int64)
    rng = np.random.default_rng(seed)

    positive_items = {}
    candidate_items = {}
    user_seen_mapped = {}
    user_hist_items = {}
    user_hist_values = {}

    dropped_unseen_test_item = {'regular': 0, 'cold': 0}
    dropped_insufficient_negatives = {'regular': 0, 'cold': 0}
    dropped_empty_mapped_history = {'cold': 0}

    eval_regular_users = []
    eval_cold_users = []

    # cache per-user history from full split train
    train_histories = {}
    for u, g in train_all.groupby('user', sort=False):
        train_histories[int(u)] = g.sort_values(['timestamp', 'item']).copy()

    for u, pos_item_raw in test_positive_raw.items():
        group = 'cold' if u in cold_users_raw else 'regular'

        if pos_item_raw not in item2idx:
            dropped_unseen_test_item[group] += 1
            continue

        pos_item_idx = item2idx[pos_item_raw]
        seen_mapped = {item2idx[i] for i in user_full_seen_raw[u] if i in item2idx}

        sampled = sample_negatives(all_items, seen_mapped, pos_item_idx, rng, num_negatives)
        if sampled is None:
            dropped_insufficient_negatives[group] += 1
            continue

        hist_df = train_histories[u]
        hist_df = hist_df[hist_df['item'].isin(item2idx)].copy()
        hist_items = hist_df['item'].map(item2idx).to_numpy(dtype=np.int64)
        hist_values = hist_df['value'].to_numpy(dtype=np.float32)

        if group == 'cold' and len(hist_items) == 0:
            dropped_empty_mapped_history['cold'] += 1
            continue

        positive_items[u] = pos_item_idx
        candidate_items[u] = sampled
        user_seen_mapped[u] = seen_mapped
        user_hist_items[u] = hist_items
        user_hist_values[u] = hist_values

        if group == 'cold':
            eval_cold_users.append(u)
        else:
            eval_regular_users.append(u)

    info = {
        'train_user_counts': train_user_counts,
        'cold_users_raw': cold_users_raw,
        'regular_users_raw': regular_users_raw,
        'num_cold_users_in_train': len(cold_users_raw),
        'num_regular_users_in_train': len(regular_users_raw),
        'dropped_unseen_test_item': dropped_unseen_test_item,
        'dropped_insufficient_negatives': dropped_insufficient_negatives,
        'dropped_empty_mapped_history': dropped_empty_mapped_history,
    }

    return {
        'train_all': train_all,
        'train_regular': train_regular,
        'user_ids': user_ids,
        'user2idx': user2idx,
        'idx2user': idx2user,
        'item_ids': item_ids,
        'item2idx': item2idx,
        'idx2item': idx2item,
        'positive_items': positive_items,
        'candidate_items': candidate_items,
        'user_seen_mapped': user_seen_mapped,
        'user_hist_items': user_hist_items,
        'user_hist_values': user_hist_values,
        'eval_regular_users': eval_regular_users,
        'eval_cold_users': eval_cold_users,
        'info': info,
    }


def metrics_to_df(dataset_name, model_name, user_group, metrics):
    return pd.DataFrame([{
        'dataset': dataset_name,
        'model': model_name,
        'user_group': user_group,
        **metrics
    }])


## 1. MovieLens

In [5]:

set_seed(SEED)

ml_ratings = pd.read_csv(BASE_DIR / '../data/ratings.csv')
ml_ratings = ml_ratings.rename(columns={
    'userId': 'user',
    'movieId': 'item',
    'rating': 'value',
    'timestamp': 'timestamp'
})

ml_ratings['user'] = ml_ratings['user'].astype(np.int64)
ml_ratings['item'] = ml_ratings['item'].astype(np.int64)
ml_ratings['value'] = ml_ratings['value'].astype(np.float32)
ml_ratings['timestamp'] = ml_ratings['timestamp'].astype(np.int64)
ml_ratings = ml_ratings.sort_values(['user', 'timestamp', 'item']).reset_index(drop=True)

ml_train_all, ml_test_positive_raw, ml_user_full_seen_raw, ml_split_meta = split_movielens_leave_one_positive(
    ml_ratings,
    rating_threshold=4.0
)

ml_data = prepare_cold_start_protocol(
    train_all=ml_train_all,
    test_positive_raw=ml_test_positive_raw,
    user_full_seen_raw=ml_user_full_seen_raw,
    num_negatives=NUM_NEGATIVES,
    seed=SEED,
)

print('MovieLens full interactions =', len(ml_ratings))
print('MovieLens split-train interactions (before cold-user removal) =', len(ml_train_all))
print('MovieLens regular-train interactions (used for training) =', len(ml_data['train_regular']))
print('MovieLens cold users in split train =', ml_data['info']['num_cold_users_in_train'])
print('MovieLens regular users in split train =', ml_data['info']['num_regular_users_in_train'])
print('MovieLens eval regular users =', len(ml_data['eval_regular_users']))
print('MovieLens eval cold users =', len(ml_data['eval_cold_users']))
print('MovieLens min split-train interactions per user =', int(ml_data['info']['train_user_counts'].min()))
print('MovieLens users with split-train interactions < 5 =', int((ml_data['info']['train_user_counts'] < COLD_USER_THRESHOLD).sum()))
print('MovieLens dropped stats:', ml_split_meta)
print('MovieLens extra dropped stats:', ml_data['info']['dropped_unseen_test_item'], ml_data['info']['dropped_insufficient_negatives'], ml_data['info']['dropped_empty_mapped_history'])


MovieLens full interactions = 1000209
MovieLens split-train interactions (before cold-user removal) = 984345
MovieLens regular-train interactions (used for training) = 984342
MovieLens cold users in split train = 1
MovieLens regular users in split train = 6037
MovieLens eval regular users = 6036
MovieLens eval cold users = 1
MovieLens min split-train interactions per user = 3
MovieLens users with split-train interactions < 5 = 1
MovieLens dropped stats: {'dropped_no_positive': 2, 'dropped_no_history_before_test': 0}
MovieLens extra dropped stats: {'regular': 1, 'cold': 0} {'regular': 0, 'cold': 0} {'cold': 0}


### 1.1 MovieLens: Popularity

In [6]:

ml_train_regular_pos = ml_data['train_regular'][ml_data['train_regular']['value'] >= 4.0].copy()

ml_pop_scores = np.zeros(len(ml_data['item_ids']), dtype=np.float32)
if len(ml_train_regular_pos) > 0:
    np.add.at(
        ml_pop_scores,
        ml_train_regular_pos['item_idx'].to_numpy(dtype=np.int64),
        np.ones(len(ml_train_regular_pos), dtype=np.float32)
    )

def ml_popularity_score_fn(users):
    return np.repeat(ml_pop_scores[None, :], repeats=len(users), axis=0)

ml_pop_regular_metrics = evaluate_sampled_ranking(
    score_fn=ml_popularity_score_fn,
    users=ml_data['eval_regular_users'],
    positive_items=ml_data['positive_items'],
    candidate_items=ml_data['candidate_items'],
    k=K,
)

ml_pop_cold_metrics = evaluate_sampled_ranking(
    score_fn=ml_popularity_score_fn,
    users=ml_data['eval_cold_users'],
    positive_items=ml_data['positive_items'],
    candidate_items=ml_data['candidate_items'],
    k=K,
)

ml_pop_results = pd.concat([
    metrics_to_df('MovieLens', 'Popularity', 'regular', ml_pop_regular_metrics),
    metrics_to_df('MovieLens', 'Popularity', 'cold_start', ml_pop_cold_metrics),
], ignore_index=True)

ml_pop_results


,dataset,model,user_group,HR@10,NDCG@10,MAP,num_eval_users
0,MovieLens,Popularity,regular,0.489563,0.270007,0.225442,6036
1,MovieLens,Popularity,cold_start,1.000000,0.386853,0.200000,1


### 1.2 MovieLens: Matrix Factorization

In [7]:

class ExplicitMF(nn.Module):
    def __init__(self, num_users, num_items, dim=64):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, dim)
        self.item_emb = nn.Embedding(num_items, dim)
        nn.init.normal_(self.user_emb.weight, std=0.05)
        nn.init.normal_(self.item_emb.weight, std=0.05)

    def forward(self, user_idx, item_idx):
        return (self.user_emb(user_idx) * self.item_emb(item_idx)).sum(dim=1)

    @torch.no_grad()
    def score_all_items_from_user_indices(self, user_indices, device):
        u = torch.as_tensor(user_indices, dtype=torch.long, device=device)
        user_vecs = self.user_emb(u)
        scores = user_vecs @ self.item_emb.weight.T
        return scores.detach().cpu().numpy()

    @torch.no_grad()
    def score_all_items_from_user_vector(self, user_vector, device):
        u = torch.as_tensor(user_vector, dtype=torch.float32, device=device).view(1, -1)
        scores = u @ self.item_emb.weight.T
        return scores.detach().cpu().numpy()[0]


ml_user_arr = ml_data['train_regular']['user_idx'].to_numpy(dtype=np.int64)
ml_item_arr = ml_data['train_regular']['item_idx'].to_numpy(dtype=np.int64)
ml_rating_arr = ml_data['train_regular']['value'].to_numpy(dtype=np.float32)

ml_model = ExplicitMF(len(ml_data['user_ids']), len(ml_data['item_ids']), dim=64).to(DEVICE)
ml_optimizer = torch.optim.Adam(ml_model.parameters(), lr=1e-3, weight_decay=1e-6)
ml_batch_size = 8192
ml_epochs = 30

start_time = time.time()

for epoch in range(1, ml_epochs + 1):
    ml_model.train()
    perm = np.random.permutation(len(ml_user_arr))
    total_loss = 0.0

    for start in range(0, len(perm), ml_batch_size):
        idx = perm[start:start + ml_batch_size]
        u = torch.as_tensor(ml_user_arr[idx], dtype=torch.long, device=DEVICE)
        i = torch.as_tensor(ml_item_arr[idx], dtype=torch.long, device=DEVICE)
        r = torch.as_tensor(ml_rating_arr[idx], dtype=torch.float32, device=DEVICE)

        pred = ml_model(u, i)
        loss = F.mse_loss(pred, r)

        ml_optimizer.zero_grad()
        loss.backward()
        ml_optimizer.step()

        total_loss += loss.item() * len(idx)

    print(f'Epoch {epoch:02d} | MovieLens MF train MSE = {total_loss / len(perm):.6f}')

print(f'MovieLens MF training time = {time.time() - start_time:.2f} seconds')


def ml_regular_mf_score_fn(users):
    user_indices = [ml_data['user2idx'][u] for u in users]
    return ml_model.score_all_items_from_user_indices(user_indices, DEVICE)


def infer_movielens_cold_user_vector(model, hist_items, hist_values, device, steps=200, lr=0.05, reg=1e-4):
    dim = model.item_emb.weight.shape[1]
    user_vec = nn.Parameter(torch.zeros(dim, device=device))

    if len(hist_items) == 0:
        return user_vec.detach().cpu().numpy()

    item_idx = torch.as_tensor(hist_items, dtype=torch.long, device=device)
    target = torch.as_tensor(hist_values, dtype=torch.float32, device=device)

    optimizer = torch.optim.Adam([user_vec], lr=lr)

    for _ in range(steps):
        item_vecs = model.item_emb(item_idx)
        pred = (item_vecs * user_vec.unsqueeze(0)).sum(dim=1)
        loss = F.mse_loss(pred, target) + reg * (user_vec.pow(2).sum())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return user_vec.detach().cpu().numpy()


def ml_cold_mf_score_fn(users):
    rows = []
    for u in users:
        hist_items = ml_data['user_hist_items'][u]
        hist_values = ml_data['user_hist_values'][u]
        u_vec = infer_movielens_cold_user_vector(
            model=ml_model,
            hist_items=hist_items,
            hist_values=hist_values,
            device=DEVICE,
            steps=200,
            lr=0.05,
            reg=1e-4,
        )
        rows.append(ml_model.score_all_items_from_user_vector(u_vec, DEVICE))
    return np.vstack(rows)


ml_mf_regular_metrics = evaluate_sampled_ranking(
    score_fn=ml_regular_mf_score_fn,
    users=ml_data['eval_regular_users'],
    positive_items=ml_data['positive_items'],
    candidate_items=ml_data['candidate_items'],
    k=K,
)

ml_mf_cold_metrics = evaluate_sampled_ranking(
    score_fn=ml_cold_mf_score_fn,
    users=ml_data['eval_cold_users'],
    positive_items=ml_data['positive_items'],
    candidate_items=ml_data['candidate_items'],
    k=K,
)

ml_mf_results = pd.concat([
    metrics_to_df('MovieLens', 'MatrixFactorization', 'regular', ml_mf_regular_metrics),
    metrics_to_df('MovieLens', 'MatrixFactorization', 'cold_start', ml_mf_cold_metrics),
], ignore_index=True)

ml_mf_results


Epoch 01 | MovieLens MF train MSE = 14.081888
Epoch 02 | MovieLens MF train MSE = 10.494262
Epoch 03 | MovieLens MF train MSE = 2.340487
Epoch 04 | MovieLens MF train MSE = 1.025926
Epoch 05 | MovieLens MF train MSE = 0.875714
Epoch 06 | MovieLens MF train MSE = 0.834596
Epoch 07 | MovieLens MF train MSE = 0.816651
Epoch 08 | MovieLens MF train MSE = 0.803859
Epoch 09 | MovieLens MF train MSE = 0.791551
Epoch 10 | MovieLens MF train MSE = 0.779176
Epoch 11 | MovieLens MF train MSE = 0.767174
Epoch 12 | MovieLens MF train MSE = 0.755976
Epoch 13 | MovieLens MF train MSE = 0.745355
Epoch 14 | MovieLens MF train MSE = 0.734938
Epoch 15 | MovieLens MF train MSE = 0.724677
Epoch 16 | MovieLens MF train MSE = 0.714435
Epoch 17 | MovieLens MF train MSE = 0.704123
Epoch 18 | MovieLens MF train MSE = 0.693682
Epoch 19 | MovieLens MF train MSE = 0.683238
Epoch 20 | MovieLens MF train MSE = 0.672829
Epoch 21 | MovieLens MF train MSE = 0.662283
Epoch 22 | MovieLens MF train MSE = 0.651807
Epoch 23

,dataset,model,user_group,HR@10,NDCG@10,MAP,num_eval_users
0,MovieLens,MatrixFactorization,regular,0.351723,0.190973,0.166433,6036
1,MovieLens,MatrixFactorization,cold_start,0.000000,0.000000,0.013514,1


## 2. Last.fm

In [8]:

set_seed(SEED)

lastfm = pd.read_csv(BASE_DIR / '../data/user_artists.dat', sep='\t')
lastfm = lastfm.rename(columns={'userID': 'user', 'artistID': 'item', 'weight': 'value'})
lastfm['user'] = lastfm['user'].astype(np.int64)
lastfm['item'] = lastfm['item'].astype(np.int64)
lastfm['value'] = lastfm['value'].astype(np.float32)

lastfm = make_random_timestamps_for_lastfm(lastfm, seed=SEED)
lastfm = lastfm.sort_values(['user', 'timestamp', 'item']).reset_index(drop=True)

lf_train_all, lf_test_positive_raw, lf_user_full_seen_raw, lf_split_meta = split_lastfm_leave_one_out(lastfm)

lf_data = prepare_cold_start_protocol(
    train_all=lf_train_all,
    test_positive_raw=lf_test_positive_raw,
    user_full_seen_raw=lf_user_full_seen_raw,
    num_negatives=NUM_NEGATIVES,
    seed=SEED,
)

print('Last.fm full interactions =', len(lastfm))
print('Last.fm split-train interactions (before cold-user removal) =', len(lf_train_all))
print('Last.fm regular-train interactions (used for training) =', len(lf_data['train_regular']))
print('Last.fm cold users in split train =', lf_data['info']['num_cold_users_in_train'])
print('Last.fm regular users in split train =', lf_data['info']['num_regular_users_in_train'])
print('Last.fm eval regular users =', len(lf_data['eval_regular_users']))
print('Last.fm eval cold users =', len(lf_data['eval_cold_users']))
print('Last.fm min split-train interactions per user =', int(lf_data['info']['train_user_counts'].min()))
print('Last.fm users with split-train interactions < 5 =', int((lf_data['info']['train_user_counts'] < COLD_USER_THRESHOLD).sum()))
print('Last.fm dropped stats:', lf_split_meta)
print('Last.fm extra dropped stats:', lf_data['info']['dropped_unseen_test_item'], lf_data['info']['dropped_insufficient_negatives'], lf_data['info']['dropped_empty_mapped_history'])


Last.fm full interactions = 92834
Last.fm split-train interactions (before cold-user removal) = 90942
Last.fm regular-train interactions (used for training) = 90920
Last.fm cold users in split train = 8
Last.fm regular users in split train = 1876
Last.fm eval regular users = 1645
Last.fm eval cold users = 5
Last.fm min split-train interactions per user = 1
Last.fm users with split-train interactions < 5 = 8
Last.fm dropped stats: {'dropped_short_history': 8}
Last.fm extra dropped stats: {'regular': 231, 'cold': 3} {'regular': 0, 'cold': 0} {'cold': 0}


### 2.1 Last.fm: Popularity

In [9]:

lf_pop_scores = np.zeros(len(lf_data['item_ids']), dtype=np.float32)
if len(lf_data['train_regular']) > 0:
    np.add.at(
        lf_pop_scores,
        lf_data['train_regular']['item_idx'].to_numpy(dtype=np.int64),
        lf_data['train_regular']['value'].to_numpy(dtype=np.float32)
    )

def lf_popularity_score_fn(users):
    return np.repeat(lf_pop_scores[None, :], repeats=len(users), axis=0)

lf_pop_regular_metrics = evaluate_sampled_ranking(
    score_fn=lf_popularity_score_fn,
    users=lf_data['eval_regular_users'],
    positive_items=lf_data['positive_items'],
    candidate_items=lf_data['candidate_items'],
    k=K,
)

lf_pop_cold_metrics = evaluate_sampled_ranking(
    score_fn=lf_popularity_score_fn,
    users=lf_data['eval_cold_users'],
    positive_items=lf_data['positive_items'],
    candidate_items=lf_data['candidate_items'],
    k=K,
)

lf_pop_results = pd.concat([
    metrics_to_df('Last.fm', 'Popularity', 'regular', lf_pop_regular_metrics),
    metrics_to_df('Last.fm', 'Popularity', 'cold_start', lf_pop_cold_metrics),
], ignore_index=True)

lf_pop_results


,dataset,model,user_group,HR@10,NDCG@10,MAP,num_eval_users
0,Last.fm,Popularity,regular,0.739818,0.510978,0.450281,1645
1,Last.fm,Popularity,cold_start,0.800000,0.412321,0.315385,5


### 2.2 Last.fm: Matrix Factorization

In [10]:

class BPRMF(nn.Module):
    def __init__(self, num_users, num_items, dim=64):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, dim)
        self.item_emb = nn.Embedding(num_items, dim)
        self.item_bias = nn.Embedding(num_items, 1)
        nn.init.normal_(self.user_emb.weight, std=0.05)
        nn.init.normal_(self.item_emb.weight, std=0.05)
        nn.init.zeros_(self.item_bias.weight)

    def score(self, user_idx, item_idx):
        return (self.user_emb(user_idx) * self.item_emb(item_idx)).sum(dim=1) + self.item_bias(item_idx).squeeze(1)

    @torch.no_grad()
    def score_all_items_from_user_indices(self, user_indices, device):
        u = torch.as_tensor(user_indices, dtype=torch.long, device=device)
        user_vecs = self.user_emb(u)
        scores = user_vecs @ self.item_emb.weight.T + self.item_bias.weight.T
        return scores.detach().cpu().numpy()

    @torch.no_grad()
    def score_all_items_from_user_vector(self, user_vector, device):
        u = torch.as_tensor(user_vector, dtype=torch.float32, device=device).view(1, -1)
        scores = u @ self.item_emb.weight.T + self.item_bias.weight.T
        return scores.detach().cpu().numpy()[0]


lf_positive_pairs = lf_data['train_regular'][['user_idx', 'item_idx']].drop_duplicates().to_numpy(dtype=np.int64)
lf_user_pos_sets = {}
for u_idx, i_idx in lf_positive_pairs:
    lf_user_pos_sets.setdefault(int(u_idx), set()).add(int(i_idx))

lf_bpr_model = BPRMF(len(lf_data['user_ids']), len(lf_data['item_ids']), dim=64).to(DEVICE)
lf_optimizer = torch.optim.Adam(lf_bpr_model.parameters(), lr=1e-3, weight_decay=1e-6)
lf_epochs = 15
lf_batch_size = 4096
rng = np.random.default_rng(SEED)

start_time = time.time()

for epoch in range(1, lf_epochs + 1):
    lf_bpr_model.train()
    perm = rng.permutation(len(lf_positive_pairs))
    epoch_loss = 0.0
    seen_examples = 0

    for start in range(0, len(perm), lf_batch_size):
        idx = perm[start:start + lf_batch_size]
        batch_pairs = lf_positive_pairs[idx]

        users_np = batch_pairs[:, 0].astype(np.int64)
        pos_np = batch_pairs[:, 1].astype(np.int64)
        neg_np = np.empty(len(batch_pairs), dtype=np.int64)

        for t, u_idx in enumerate(users_np):
            while True:
                j = int(rng.integers(0, len(lf_data['item_ids'])))
                if j not in lf_user_pos_sets[u_idx]:
                    neg_np[t] = j
                    break

        u = torch.as_tensor(users_np, dtype=torch.long, device=DEVICE)
        i = torch.as_tensor(pos_np, dtype=torch.long, device=DEVICE)
        j = torch.as_tensor(neg_np, dtype=torch.long, device=DEVICE)

        pos_scores = lf_bpr_model.score(u, i)
        neg_scores = lf_bpr_model.score(u, j)
        loss = -F.logsigmoid(pos_scores - neg_scores).mean()

        lf_optimizer.zero_grad()
        loss.backward()
        lf_optimizer.step()

        epoch_loss += loss.item() * len(batch_pairs)
        seen_examples += len(batch_pairs)

    print(f'Epoch {epoch:02d} | Last.fm BPR-MF train loss = {epoch_loss / seen_examples:.6f}')

print(f'Last.fm BPR-MF training time = {time.time() - start_time:.2f} seconds')


def lf_regular_mf_score_fn(users):
    user_indices = [lf_data['user2idx'][u] for u in users]
    return lf_bpr_model.score_all_items_from_user_indices(user_indices, DEVICE)


def infer_lastfm_cold_user_vector(model, hist_items, num_items, device, steps=200, lr=0.05, reg=1e-4, seed=42):
    rng_local = np.random.default_rng(seed)
    dim = model.item_emb.weight.shape[1]
    user_vec = nn.Parameter(torch.zeros(dim, device=device))

    if len(hist_items) == 0:
        return user_vec.detach().cpu().numpy()

    pos_set = set(int(x) for x in hist_items.tolist())
    pos_idx = torch.as_tensor(hist_items, dtype=torch.long, device=device)
    optimizer = torch.optim.Adam([user_vec], lr=lr)

    for _ in range(steps):
        neg_samples = []
        for _p in hist_items:
            while True:
                j = int(rng_local.integers(0, num_items))
                if j not in pos_set:
                    neg_samples.append(j)
                    break

        neg_idx = torch.as_tensor(neg_samples, dtype=torch.long, device=device)
        pos_vecs = model.item_emb(pos_idx)
        neg_vecs = model.item_emb(neg_idx)
        pos_bias = model.item_bias(pos_idx).squeeze(1)
        neg_bias = model.item_bias(neg_idx).squeeze(1)

        pos_scores = (pos_vecs * user_vec.unsqueeze(0)).sum(dim=1) + pos_bias
        neg_scores = (neg_vecs * user_vec.unsqueeze(0)).sum(dim=1) + neg_bias

        loss = -F.logsigmoid(pos_scores - neg_scores).mean() + reg * (user_vec.pow(2).sum())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    return user_vec.detach().cpu().numpy()


def lf_cold_mf_score_fn(users):
    rows = []
    for u in users:
        hist_items = lf_data['user_hist_items'][u]
        u_vec = infer_lastfm_cold_user_vector(
            model=lf_bpr_model,
            hist_items=hist_items,
            num_items=len(lf_data['item_ids']),
            device=DEVICE,
            steps=200,
            lr=0.05,
            reg=1e-4,
            seed=SEED + int(u),
        )
        rows.append(lf_bpr_model.score_all_items_from_user_vector(u_vec, DEVICE))
    return np.vstack(rows)


lf_mf_regular_metrics = evaluate_sampled_ranking(
    score_fn=lf_regular_mf_score_fn,
    users=lf_data['eval_regular_users'],
    positive_items=lf_data['positive_items'],
    candidate_items=lf_data['candidate_items'],
    k=K,
)

lf_mf_cold_metrics = evaluate_sampled_ranking(
    score_fn=lf_cold_mf_score_fn,
    users=lf_data['eval_cold_users'],
    positive_items=lf_data['positive_items'],
    candidate_items=lf_data['candidate_items'],
    k=K,
)

lf_mf_results = pd.concat([
    metrics_to_df('Last.fm', 'MatrixFactorization', 'regular', lf_mf_regular_metrics),
    metrics_to_df('Last.fm', 'MatrixFactorization', 'cold_start', lf_mf_cold_metrics),
], ignore_index=True)

lf_mf_results


Epoch 01 | Last.fm BPR-MF train loss = 0.689984
Epoch 02 | Last.fm BPR-MF train loss = 0.679806
Epoch 03 | Last.fm BPR-MF train loss = 0.670140
Epoch 04 | Last.fm BPR-MF train loss = 0.660095
Epoch 05 | Last.fm BPR-MF train loss = 0.649145
Epoch 06 | Last.fm BPR-MF train loss = 0.636008
Epoch 07 | Last.fm BPR-MF train loss = 0.619535
Epoch 08 | Last.fm BPR-MF train loss = 0.597561
Epoch 09 | Last.fm BPR-MF train loss = 0.568955
Epoch 10 | Last.fm BPR-MF train loss = 0.533701
Epoch 11 | Last.fm BPR-MF train loss = 0.493482
Epoch 12 | Last.fm BPR-MF train loss = 0.452295
Epoch 13 | Last.fm BPR-MF train loss = 0.412464
Epoch 14 | Last.fm BPR-MF train loss = 0.375877
Epoch 15 | Last.fm BPR-MF train loss = 0.344087
Last.fm BPR-MF training time = 12.78 seconds


,dataset,model,user_group,HR@10,NDCG@10,MAP,num_eval_users
0,Last.fm,MatrixFactorization,regular,0.799392,0.602422,0.548599,1645
1,Last.fm,MatrixFactorization,cold_start,0.600000,0.526186,0.516204,5


## 3. Final Results

In [11]:

all_results = pd.concat([
    ml_pop_results,
    ml_mf_results,
    lf_pop_results,
    lf_mf_results,
], ignore_index=True)

all_results


,dataset,model,user_group,HR@10,NDCG@10,MAP,num_eval_users
0,MovieLens,Popularity,regular,0.489563,0.270007,0.225442,6036
1,MovieLens,Popularity,cold_start,1.000000,0.386853,0.200000,1
2,MovieLens,MatrixFactorization,regular,0.351723,0.190973,0.166433,6036
3,MovieLens,MatrixFactorization,cold_start,0.000000,0.000000,0.013514,1
4,Last.fm,Popularity,regular,0.739818,0.510978,0.450281,1645
5,Last.fm,Popularity,cold_start,0.800000,0.412321,0.315385,5
6,Last.fm,MatrixFactorization,regular,0.799392,0.602422,0.548599,1645
7,Last.fm,MatrixFactorization,cold_start,0.600000,0.526186,0.516204,5
